# Task 2 — Multiple operations with Matrices using multithreading

**6CS005 High Performance Computing | Arkhan Shimar**

The program reads consecutive matrix pairs using dynamic allocation. Each valid operation distributes output rows with OpenMP. The requested thread count is capped to the number of output rows. Invalid operations are reported individually; zero divisors produce NaN.

Run the cells from top to bottom. A standard CPU runtime is sufficient. No Google Drive mount or manual input upload is required. The final cell downloads the real run outputs.

The application is C/CUDA; Python cells only arrange files, validate results, and display images.

## Get the public project files

The exact source revision is pinned so later repository edits cannot silently change these inputs.

In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/hpc_task2
curl --fail --location --retry 3 https://codeload.github.com/ArkhanShimar/HPC/tar.gz/5fca2bacfa3490680292677c7514be49755bd1f1 -o /content/hpc_task2.tar.gz
tar -xzf /content/hpc_task2.tar.gz -C /content/hpc_task2 --strip-components=1
echo "Project revision: 5fca2bacfa3490680292677c7514be49755bd1f1"

In [ ]:
%cd /content/hpc_task2/Task2

## Runtime details

In [ ]:
%%bash
set -euo pipefail
{
date -u
gcc --version | head -1
lscpu | grep -E "Model name|CPU\(s\)|Thread|Core"
} | tee environment.txt

## C source

In [ ]:
%%writefile matrix_operations.c
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <limits.h>
#include <ctype.h>
#include <errno.h>
#include <math.h>
#include <omp.h>

typedef struct { int rows, cols; double *data; } Matrix;
static size_t line_number;
static double compute_seconds;

// Read a physical line so a missing value cannot be taken from the next row.
static int read_line(FILE *fp, char **out) {
    size_t used = 0, capacity = 128;
    char *line = malloc(capacity), c;
    if (!line) return -1;
    int rc;
    while ((rc = fscanf(fp, "%c", &c)) == 1 && c != '\n') {
        if (used + 1 == capacity) {
            if (capacity > SIZE_MAX / 2) { free(line); return -1; }
            char *larger = realloc(line, capacity * 2);
            if (!larger) { free(line); return -1; }
            line = larger; capacity *= 2;
        }
        line[used++] = c;
    }
    if (ferror(fp)) { free(line); return -1; }
    if (rc != 1 && !used) { free(line); return 0; }
    line[used] = '\0'; *out = line; line_number++; return 1;
}
static char *skip_space(char *p) { while (isspace((unsigned char)*p)) p++; return p; }
static int allocate(Matrix *m, int rows, int cols) {
    if (rows <= 0 || cols <= 0 || (size_t)rows > SIZE_MAX / (size_t)cols / sizeof(double)) return 0;
    m->rows = rows; m->cols = cols;
    m->data = malloc((size_t)rows * cols * sizeof(double));
    return m->data != NULL;
}
static int read_matrix(FILE *fp, Matrix *m) {
    char *line = NULL, *p, *end; int rc;
    do {
        free(line); line = NULL; rc = read_line(fp, &line);
        if (rc <= 0) return rc;
        p = skip_space(line);
    } while (!*p);
    errno = 0; long rows = strtol(p, &end, 10);
    if (p == end || errno || rows <= 0 || rows > INT_MAX) goto invalid;
    p = skip_space(end); if (*p++ != ',') goto invalid;
    p = skip_space(p); errno = 0; long cols = strtol(p, &end, 10);
    if (p == end || errno || cols <= 0 || cols > INT_MAX || *skip_space(end)) goto invalid;
    free(line); line = NULL;
    if (!allocate(m, (int)rows, (int)cols)) {
        fprintf(stderr, "Line %zu: matrix too large or allocation failed.\n", line_number); return -1;
    }
    for (int r = 0; r < m->rows; ++r) {
        rc = read_line(fp, &line);
        if (rc != 1) { fprintf(stderr, "Missing row %d after line %zu.\n", r+1, line_number); return -1; }
        p = line;
        for (int c = 0; c < m->cols; ++c) {
            p = skip_space(p); errno = 0;
            double value = strtod(p, &end);
            if (p == end || errno || !isfinite(value)) goto invalid;
            m->data[(size_t)r*m->cols+c] = value;
            p = skip_space(end);
            if (c + 1 < m->cols) { if (*p != ',') goto invalid; p++; }
            else if (*p) goto invalid;
        }
        free(line); line = NULL;
    }
    return 1;
invalid:
    fprintf(stderr, "Line %zu: invalid header, numeric value or row length.\n", line_number);
    free(line); return -1;
}
static void write_matrix(FILE *out, const char *name, const Matrix *m) {
    fprintf(out, "%s - %d,%d\n", name, m->rows, m->cols);
    for (int r = 0; r < m->rows; ++r) {
        for (int c = 0; c < m->cols; ++c) {
            double v = m->data[(size_t)r*m->cols+c];
            if (isnan(v)) fprintf(out, "NaN"); else fprintf(out, "%.12g", v);
            fprintf(out, "%s", c+1 == m->cols ? "\n" : ",");
        }
    }
    fprintf(out, "\n");
}
static int operation(FILE *out, const Matrix *a, const Matrix *b, int op, int requested) {
    const char *names[] = {"Addition", "Subtraction", "Element-wise multiplication", "Element-wise division", "Transpose A", "Transpose B", "Matrix multiplication"};
    if (op < 4 && (a->rows != b->rows || a->cols != b->cols)) {
        fprintf(out, "%s cannot be done (shapes differ).\n", names[op]); return 1;
    }
    if (op == 6 && a->cols != b->rows) {
        fprintf(out, "%s cannot be done (A.cols != B.rows).\n", names[op]); return 1;
    }
    const Matrix *source = op == 5 ? b : a;
    int rows = op == 4 || op == 5 ? source->cols : a->rows;
    int cols = op == 4 || op == 5 ? source->rows : (op == 6 ? b->cols : a->cols);
    Matrix result = {0};
    if (!allocate(&result, rows, cols)) { fprintf(stderr, "Result allocation failed.\n"); return 0; }
    int threads = requested < rows ? requested : rows;
    double start = omp_get_wtime();
    // Each iteration owns one complete output row; no shared accumulator is needed.
    #pragma omp parallel for num_threads(threads) schedule(static)
    for (int r = 0; r < rows; ++r) {
        for (int c = 0; c < cols; ++c) {
            size_t i = (size_t)r*cols+c;
            if (op == 0) result.data[i] = a->data[i] + b->data[i];
            else if (op == 1) result.data[i] = a->data[i] - b->data[i];
            else if (op == 2) result.data[i] = a->data[i] * b->data[i];
            else if (op == 3) result.data[i] = b->data[i] == 0 ? NAN : a->data[i] / b->data[i];
            else if (op == 4 || op == 5) result.data[i] = source->data[(size_t)c*source->cols+r];
            else {
                double sum = 0;
                for (int k = 0; k < a->cols; ++k) sum += a->data[(size_t)r*a->cols+k] * b->data[(size_t)k*b->cols+c];
                result.data[i] = sum;
            }
        }
    }
    compute_seconds += omp_get_wtime() - start;
    printf("  %s: %dx%d, thread limit %d\n", names[op], rows, cols, threads);
    write_matrix(out, names[op], &result); free(result.data);
    return !ferror(out);
}
int main(int argc, char **argv) {
    if (argc != 3) { fprintf(stderr, "Usage: %s input.txt threads\n", argv[0]); return 1; }
    char *end; errno = 0; long threads = strtol(argv[2], &end, 10);
    if (errno || end == argv[2] || *end || threads < 1 || threads > INT_MAX) {
        fprintf(stderr, "Thread count must be a positive integer.\n"); return 1;
    }
    FILE *input = fopen(argv[1], "r");
    if (!input) { perror(argv[1]); return 1; }
    FILE *out = fopen("results.txt.tmp", "w");
    if (!out) { perror("results.txt.tmp"); fclose(input); return 1; }
    Matrix a = {0}, b = {0}; int status = 1, pairs = 0, rc;
    omp_set_dynamic(0);
    while ((rc = read_matrix(input, &a)) == 1) {
        if (read_matrix(input, &b) != 1) { fprintf(stderr, "Incomplete or invalid matrix pair %d.\n", pairs+1); goto cleanup; }
        pairs++;
        fprintf(out, "Pair %d | A: %d,%d | B: %d,%d\n\n", pairs, a.rows, a.cols, b.rows, b.cols);
        printf("Pair %d: A %dx%d, B %dx%d\n", pairs, a.rows, a.cols, b.rows, b.cols);
        for (int op = 0; op < 7; ++op) if (!operation(out, &a, &b, op, (int)threads)) goto cleanup;
        free(a.data); free(b.data); a.data = b.data = NULL;
    }
    if (rc < 0 || !pairs) { fprintf(stderr, "Input does not contain complete valid matrix pairs.\n"); goto cleanup; }
    if (ferror(out)) goto cleanup;
    status = 0;
cleanup:
    free(a.data); free(b.data);
    if (fclose(input)) status = 1;
    if (fclose(out)) status = 1;
    if (!status) {
        if (rename("results.txt.tmp", "results.txt")) { perror("results.txt"); status = 1; }
        else printf("Processed %d pairs. Compute time: %.6f s. Saved results.txt\n", pairs, compute_seconds);
    }
    if (status) remove("results.txt.tmp");
    return status;
}


## Compile

In [ ]:
%%bash
set -euo pipefail
gcc -std=c11 -O2 -Wall -Wextra -Wpedantic -fopenmp matrix_operations.c -lm -o matrix_operations

## Run the main example

In [ ]:
%%bash
set -euo pipefail
{
./matrix_operations ../data/MatData.txt 4
printf "\nFirst pair output excerpt\n"
head -48 results.txt
} | tee run.log

## Correctness checks and measurements

The checks compare against independent CPU references and exercise invalid input. Five measured repeats follow a warm-up where appropriate. CPU timings cover the computation stated by each program; CUDA event timings cover kernels only, not file access, allocation or transfers. These scopes must not be compared as end-to-end speedups.

In [ ]:
%%bash
set -euo pipefail
python ../tests/verify.py 2 | tee validation.log

## Viva discussion

Explain valid matrix shapes, row ownership, private accumulators, the implicit barrier, thread capping, NaN division results, and why parsing one physical row at a time matters.

## Save the evidence from this run

This downloads output files and validation logs. To keep notebook output cells too, choose **File → Download → Download .ipynb** after the run. A fresh viewer can rerun the same public notebook.

In [ ]:
from pathlib import Path
import shutil, zipfile
from google.colab import files
archive = Path('/content/Task2_outputs.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for pattern in ['*.txt', '*.log', '*.png', 'validation/*.json']:
        for path in Path('.').glob(pattern):
            z.write(path, path.as_posix())
files.download(str(archive))